<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook spot-checks physical phasing from HiPhase. For an example sample, it compares unphased and phased VCF genotypes at overlapping sites and prints positions where the phased alleles differ from the original call.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
# Spot-check that HiPhase phasing does not change genotypes: for each site, compare the unphased
# genotype with the phased one, ignoring allele order. Both files must be local.
import gzip

In [ ]:
import os

# Location of this notebook's data (a local directory). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Choose one pair of files: small variants for one sample (this cell) or SVs for HG002 (next cell).
# Running both leaves the second pair selected.
original_filename = f"{AOU_LR_DATA}/example_sample.subset.g.vcf.bgz"
phased_file = f"{AOU_LR_DATA}/example_sample_phased_snp.sorted.vcf.gz"

In [ ]:
original_filename = f"{AOU_LR_DATA}/HG002_sv_cleaned.vcf.gz"
phased_file = f"{AOU_LR_DATA}/HG002_phased_sv.sorted.vcf.gz"

In [ ]:
# Unphased genotype (first sample column) at each position.
with gzip.open(original_filename, 'r') as fp:
    original = fp.readlines()

In [ ]:
Original = {}
for line in original:
    line = line.decode()
    if line.startswith("#"):
        continue
    itemlist = line.split('\t')
    pos = itemlist[1]
    genotype = itemlist[9].split(':')[0].split("/")
    Original[pos] = genotype


In [ ]:
with gzip.open(phased_file, 'r') as fp:
    phased = fp.readlines()

In [ ]:
# Print any site where the phased genotype has different alleles from the unphased one.
for line in phased:
    line = line.decode()
    if line.startswith("#"):
        continue
    itemlist = line.split('\t')
    pos = itemlist[1]
    genotype = itemlist[9].split(':')[0]
    if "/" in genotype:
        g = genotype.split('/')
    else:
        g = genotype.split('|')
    if sorted(g) != sorted(Original[pos]):
        print(pos, g, Original[pos])